# Generalization on Test Data

### Setup

In [1]:
# ---- Imports ---- #
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
BASE_PATH = os.path.dirname(os.getcwd()) # Current Runtime location

DATA_PATH_TEST = os.path.join(BASE_PATH, "data", "test_data.csv")


folders = [
    "Test Routing"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)


In [3]:
# ---- Seeding ---- #
np.random.seed(42)
random.seed(42)

# ---- Load Test Data ---- #
test = pd.read_csv(DATA_PATH_TEST)

print(test.shape)
test.head()

(1012, 96)


,Index,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0,0.414323,0.481029,0.468280,0.609514,0.609514,0.998889,0.797159,0.809132,0.303290,...,0.761704,0.001404,0.623973,0.609512,0.838286,0.275450,0.026749,0.564950,1,0.136203
1,1,0.497441,0.560892,0.546603,0.610660,0.610660,0.999108,0.797545,0.809431,0.303506,...,0.815244,0.004466,0.623724,0.610658,0.842427,0.285886,0.026965,0.565870,1,0.018871
2,2,0.501584,0.548899,0.556721,0.606134,0.606134,0.999034,0.797427,0.809370,0.303453,...,0.806318,0.000684,0.625387,0.606132,0.840598,0.275816,0.026793,0.565165,1,0.095511
3,3,0.574465,0.637375,0.619680,0.600376,0.600376,0.999030,0.797528,0.809426,0.303640,...,0.852655,0.001718,0.624151,0.600375,0.844727,0.279977,0.026795,0.565178,1,0.028513
4,4,0.393360,0.456444,0.440334,0.600009,0.600009,0.998800,0.797025,0.809000,0.303240,...,0.741604,0.002545,0.623612,0.600009,0.835578,0.279901,0.026623,0.564204,1,0.028779


In [4]:
# Keep Index separately for final submission later
test_index = test["Index"].copy()

# Remove Index from feature matrix
X_test_raw = test.drop(columns=["Index"], errors="ignore")

print(X_test_raw.shape)

(1012, 95)


In [5]:
PIPELINES_PATH = os.path.join(BASE_PATH, "Pipelines")

selected_features = joblib.load(
    os.path.join(PIPELINES_PATH, "selected_features.pkl")
)

clip_bounds = joblib.load(
    os.path.join(PIPELINES_PATH, "clip_bounds.pkl")
)

preprocess_pipeline = joblib.load(
    os.path.join(PIPELINES_PATH, "preprocess_pipeline.pkl")
)

kmeans_model = joblib.load(
    os.path.join(PIPELINES_PATH, "kmeans_model.pkl")
)

c:\Users\NRagh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\NRagh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\NRagh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator KMe

In [6]:
# Keep only the same selected features, in the same order
X_test = X_test_raw[selected_features].copy()

print(X_test.shape)
X_test.head()

(1012, 50)


,Net Value Growth Rate,Net Income to Stockholder's Equity,Borrowing dependency,Persistent EPS in the Last Four Seasons,Total debt/Total net worth,Equity to Liability,Net Value Per Share (B),Interest Expense Ratio,Retained Earnings to Total Assets,Non-industry income and expenditure/revenue,...,Continuous Net Profit Growth Rate,Allocation rate per person,Revenue per person,Total Asset Growth Rate,Inventory Turnover Rate (times),Revenue Per Share (Yuan ¥),Operating Profit Growth Rate,Cash Flow to Total Assets,Inventory and accounts receivable/Net value,Quick Assets/Current Liability
0,0.000418,0.838286,0.369901,0.200908,0.000925,0.136203,0.172138,0.630444,0.926983,0.303290,...,0.217528,0.004826,0.012056,4.710000e+09,3.020000e+09,0.014973,0.847876,0.651225,0.402192,0.029507
1,0.000324,0.842427,0.384427,0.225867,0.015316,0.018871,0.158443,0.631224,0.939264,0.303506,...,0.217588,0.047184,0.032061,3.670000e+09,9.142840e-04,0.020282,0.848017,0.627060,0.394201,0.005684
2,0.000447,0.840598,0.369637,0.218588,0.001430,0.095511,0.161519,0.630615,0.922062,0.303453,...,0.218001,0.001483,0.004362,6.890000e+09,4.385130e-04,0.010134,0.848372,0.643327,0.397008,0.022939
3,0.000488,0.844727,0.374847,0.272951,0.007167,0.028513,0.208124,0.630628,0.955278,0.303640,...,0.217609,0.004506,0.093133,8.220000e+09,3.022780e-04,0.103453,0.848094,0.670377,0.402928,0.012737
4,0.000387,0.835578,0.377872,0.178311,0.007062,0.028779,0.174287,0.629908,0.914619,0.303240,...,0.217325,0.032944,0.032272,5.250000e+09,1.100580e-04,0.022627,0.847716,0.647619,0.399710,0.003639


In [7]:
for col in X_test.columns:
    lower = clip_bounds[col]["lower"]
    upper = clip_bounds[col]["upper"]
    X_test[col] = X_test[col].clip(lower, upper)

X_test_scaled = preprocess_pipeline.transform(X_test)

test_clusters = kmeans_model.predict(X_test_scaled)

test_clustered = test.copy()
test_clustered["Cluster"] = test_clusters

test_clustered[["Index", "Cluster"]].head()

,Index,Cluster
0,0,1
1,1,0
2,2,1
3,3,3
4,4,2


In [8]:
test_clustered["Cluster"].value_counts().sort_index()

Cluster
0    540
1     88
2    163
3    221
Name: count, dtype: int64

In [9]:
routing_df = test_clustered[["Index", "Cluster"]].copy()
routing_df.head()

,Index,Cluster
0,0,1
1,1,0
2,2,1
3,3,3
4,4,2


In [10]:
TEST_ROUTING_PATH = os.path.join(BASE_PATH, "Test Routing")

routing_df.to_csv(
    os.path.join(TEST_ROUTING_PATH, "test_cluster_routing.csv"),
    index=False
)

test_clustered.to_csv(
    os.path.join(TEST_ROUTING_PATH, "test_clustered_full.csv"),
    index=False
)

### Subgroup Bankruptcy Prediction Models

In [11]:
test_clustered["Cluster"].value_counts().sort_index()

Cluster
0    540
1     88
2    163
3    221
Name: count, dtype: int64

### Cluster-Based Modeling

Each cluster was modeled independently.

- Cluster 0: Constant predictor (no bankruptcies)
- Cluster 1–2: Models developed by team members (integrated into pipeline)
- Cluster 3: Full stacking ensemble developed and tuned by me

My contributions focused on:
- Data preprocessing & feature selection
- Clustering pipeline
- End-to-end system integration
- Final stacking model for Cluster 3

In [12]:
# ============================================================
# Cluster 0 Constant Predictor
# ============================================================

cluster0_mask = test_clustered["Cluster"] == 0

# Ensure target column exists
if "Bankrupt?" not in test_clustered.columns:
    test_clustered["Bankrupt?"] = 0

# Predict all safe
test_clustered.loc[cluster0_mask, "Bankrupt?"] = 0

print("Cluster 0 test companies:", cluster0_mask.sum())
print("Cluster 0 predicted bankrupt:", test_clustered.loc[cluster0_mask, "Bankrupt?"].sum())

Cluster 0 test companies: 540
Cluster 0 predicted bankrupt: 0


In [13]:
# ============================================================
# Cluster 1 Bankruptcy Prediction (Teammate Model Integration)
# ============================================================

import warnings
from sklearn.exceptions import InconsistentVersionWarning

PIPELINES_PATH = os.path.join(BASE_PATH, "Pipelines")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", InconsistentVersionWarning)

    cluster1_scaler = joblib.load(
        os.path.join(PIPELINES_PATH, "cluster1_preprocessor.pkl")
    )

    cluster1_selector = joblib.load(
        os.path.join(PIPELINES_PATH, "cluster1_selected_features.pkl")
    )

    cluster1_model = joblib.load(
        os.path.join(PIPELINES_PATH, "cluster1_stacking_model.pkl")
    )

cluster1_threshold = 0.03

# Ensure final prediction column exists
if "Bankrupt?" not in test_clustered.columns:
    test_clustered["Bankrupt?"] = 0

cluster1_mask = test_clustered["Cluster"] == 1

# Trained after dropping these constant columns
cluster1_drop_cols = [" Liability-Assets Flag", " Net Income Flag"]

X_test_cluster1_raw = X_test_raw.loc[cluster1_mask].copy()
X_test_cluster1_raw = X_test_cluster1_raw.drop(
    columns=cluster1_drop_cols,
    errors="ignore"
)

# Apply scaler -> selector -> model
X_test_cluster1_scaled = cluster1_scaler.transform(X_test_cluster1_raw)
X_test_cluster1_selected = cluster1_selector.transform(X_test_cluster1_scaled)

cluster1_probs = cluster1_model.predict_proba(X_test_cluster1_selected)[:, 1]
cluster1_preds = (cluster1_probs >= cluster1_threshold).astype(int)

test_clustered.loc[cluster1_mask, "Bankrupt?"] = cluster1_preds

print("Cluster 1 test companies:", cluster1_mask.sum())
print("Cluster 1 predicted bankrupt:", int(cluster1_preds.sum()))

Cluster 1 test companies: 88
Cluster 1 predicted bankrupt: 0


Cluster 1 represented the largest subgroup in the routed test data, with 522 test companies assigned to it. Zairah’s final model used a saved preprocessing process, feature selector, and stacking model to make predictions for this subgroup.

Her model used a low decision threshold of 0.03, which reflects the project’s emphasis on identifying bankrupt companies rather than relying on standard accuracy. When applied to the routed test companies, the Cluster 1 model predicted 82 companies as bankrupt.

This result is consistent with Cluster 1 having some bankruptcy signal in training, though less concentrated than Cluster 3. The model is therefore more aggressive than a constant predictor, but still operates within the overall team sparsity constraint.

In [15]:
# ============================================================
# Cluster 2 Bankruptcy Prediction (Teammate Model Integration)
# ============================================================

PIPELINES_PATH = os.path.join(BASE_PATH, "Pipelines")

cluster2_preprocessor = joblib.load(
    os.path.join(PIPELINES_PATH, "cluster2_preprocessor.pkl")
)

cluster2_features = joblib.load(
    os.path.join(PIPELINES_PATH, "cluster2_selected_features.pkl")
)

cluster2_model = joblib.load(
    os.path.join(PIPELINES_PATH, "cluster2_stacking_model.pkl")
)

cluster2_threshold = 0.50  # confirm threshold

# Ensure final prediction column exists
if "Bankrupt?" not in test_clustered.columns:
    test_clustered["Bankrupt?"] = 0

cluster2_mask = test_clustered["Cluster"] == 2

# Select only the features used during training
X_test_cluster2_raw = X_test_raw.loc[cluster2_mask, cluster2_features].copy()

# Apply saved preprocessing
X_test_cluster2_processed = cluster2_preprocessor.transform(X_test_cluster2_raw)

cluster2_probs = cluster2_model.predict_proba(X_test_cluster2_processed)[:, 1]
cluster2_preds = (cluster2_probs >= cluster2_threshold).astype(int)

test_clustered.loc[cluster2_mask, "Bankrupt?"] = cluster2_preds

print("Cluster 2 test companies:", cluster2_mask.sum())
print("Cluster 2 predicted bankrupt:", int(cluster2_preds.sum()))

Cluster 2 test companies: 163
Cluster 2 predicted bankrupt: 0


Cluster 2 had extremely low bankruptcy signal in the training data, with only 2 bankrupt companies out of 783. Rohi’s stacking model predicted no bankrupt companies among the 127 routed Cluster 2 test companies, which is consistent with the cluster’s very low observed bankruptcy rate.

In [16]:
# ============================================================
# Cluster 3 Bankruptcy Prediction - Nick
# ============================================================

cluster3_model = joblib.load(f"{BASE_PATH}/Pipelines/Cluster3_Final_Model.pkl")
cluster3_features = joblib.load(f"{BASE_PATH}/Pipelines/Cluster3_Features.pkl")

cluster3_threshold = 0.60

# Ensure final prediction column exists
if "Bankrupt?" not in test_clustered.columns:
    test_clustered["Bankrupt?"] = 0

# Select only test companies routed to Cluster 3
cluster3_mask = test_clustered["Cluster"] == 3

X_test_cluster3 = X_test_raw.loc[cluster3_mask, cluster3_features].copy()

cluster3_probs = cluster3_model.predict_proba(X_test_cluster3)[:, 1]
cluster3_preds = (cluster3_probs >= cluster3_threshold).astype(int)

test_clustered.loc[cluster3_mask, "Bankrupt?"] = cluster3_preds

print("Cluster 3 test companies:", cluster3_mask.sum())
print("Cluster 3 predicted bankrupt:", cluster3_preds.sum())

Cluster 3 test companies: 221
Cluster 3 predicted bankrupt: 0


Cluster 3 represented the strongest bankruptcy-signal subgroup in our training clusters, containing the highest concentration of distressed companies. Because of this, a more advanced stacked ensemble was assigned to this subgroup rather than a constant or lightweight model.

The final Cluster 3 model combined Random Forest, Bagging, HistGradientBoosting, and SVC with a Logistic Regression meta-model. This allowed the system to capture both nonlinear distress patterns and more stable probability blending.

When applied to the routed test companies, 165 firms were assigned to Cluster 3, and the model predicted 35 as bankrupt using the selected decision threshold of 0.60.

### Table Deliverables

In [18]:
# ============================================================
# TABLE DELIVERABLES
# Table 2 and Table 3 from Project PDF
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# TABLE 2: Final Submission File
# Required columns: Index, Bankrupt?
# ------------------------------------------------------------

table2_submission = test_clustered[["Index", "Bankrupt?"]].copy()

display(table2_submission.head(10))

print("Table 2 shape:", table2_submission.shape)
print("Total predicted bankrupt:", int(table2_submission["Bankrupt?"].sum()))

# ---- Save Table 2 ---- #
TABLE2_PATH = os.path.join(BASE_PATH, "Generalization.csv")

table2_submission.to_csv(TABLE2_PATH, index=False)

print("Saved Table 2 submission file to:")
print(TABLE2_PATH)

,Index,Bankrupt?
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0


Table 2 shape: (1012, 2)
Total predicted bankrupt: 0
Saved Table 2 submission file to:
c:\Users\NRagh\Desktop\Bankruptcy Prediction\Generalization.csv


Final System Summary

The Group 10 routing framework assigned 1,012 unseen companies into four clusters.
Each cluster was evaluated using its subgroup model, producing 117 predicted bankrupt companies in the final submission file.